# Nigerian Political Parties Sentiment Analysis

Simple sentiment analysis of **recent Twitter** mentions for Nigeria's three major political parties:

- **APC** (All Progressives Congress)
- **PDP** (People's Democratic Party)
- **LP** (Labour Party)

Uses a pre-trained sentiment model and a **Gradio UI** to explore results by party and analyze custom text.

**QLoRA fine-tuning:** used to fine-tune the model on your data with 4-bit quantization + LoRA; then load the adapters and use the fine-tuned model in the app.

In [ ]:
# Install dependencies (run once)
# For QLoRA fine-tuning also need: peft, bitsandbytes, accelerate, datasets, scikit-learn
%pip install -q gradio transformers torch pandas tweepy python-dotenv peft bitsandbytes accelerate datasets scikit-learn

In [ ]:
import os
import pandas as pd
from dotenv import load_dotenv
from transformers import pipeline
import gradio as gr

load_dotenv()

In [ ]:
import tweepy

# Search queries per party (recent tweets from Nigeria / Nigerian politics)
PARTY_QUERIES = {
    "APC": "APC Nigeria OR Tinubu APC -is:retweet lang:en",
    "PDP": "PDP Nigeria OR Atiku PDP -is:retweet lang:en",
    "LP": "Labour Party Nigeria OR Peter Obi LP -is:retweet lang:en",
}

def fetch_tweets_twitter(max_per_party: int = 50) -> pd.DataFrame | None:
    bearer_token = os.getenv("TWITTER_BEARER_TOKEN") or os.getenv("BEARER_TOKEN")
    if not bearer_token:
        print("No TWITTER_BEARER_TOKEN in .env")
        return None
    try:
        client = tweepy.Client(bearer_token=bearer_token)
        rows = []
        for party, query in PARTY_QUERIES.items():
            try:
                resp = client.search_recent_tweets(
                    query=query,
                    max_results=min(max_per_party, 100),
                    tweet_fields=["created_at", "text"],
                    user_fields=[],
                    expansions=[],
                )
                if resp.data:
                    for t in resp.data:
                        text = (t.text or "").strip()
                        if text:
                            rows.append({"party": party, "text": text})
            except tweepy.TweepyException as e:
                print(f"Twitter API ({party}): {e}")
        if not rows:
            print("No tweets returned")
            return None
        print(f"Fetched {len(rows)} tweets from Twitter.")
        return pd.DataFrame(rows)
    except Exception as e:
        print(f"Tweepy error: {e}. Using sample data.")
        return None


df_tweets = fetch_tweets_twitter()

In [ ]:
sentiment_pipeline = pipeline(
    "sentiment-analysis",
    model="cardiffnlp/twitter-roberta-base-sentiment-latest",
    truncation=True,
    max_length=128,
)

In [ ]:
def get_sentiment(text: str) -> str:
    """Get sentiment label for a single text (truncated)."""
    out = sentiment_pipeline(text[:512], truncation=True, max_length=128)
    return out[0]["label"].lower()

def analyze_corpus(tweets_df: pd.DataFrame) -> pd.DataFrame:
    """Run sentiment on all tweets and return DataFrame with party and sentiment."""
    tweets_df = tweets_df.copy()
    tweets_df["sentiment"] = tweets_df["text"].apply(get_sentiment)
    return tweets_df

# Run sentiment on sample data
df_analyzed = analyze_corpus(df_tweets)
df_analyzed.head(10)

In [ ]:
# Summary counts by party and sentiment
summary = df_analyzed.groupby(["party", "sentiment"]).size().unstack(fill_value=0)
summary

## QLoRA fine-tuning

Fine-tune the sentiment model on Nigerian political tweets using **QLoRA** (4-bit quantization + LoRA) so it adapts to local wording and context. Run the cells below to train and save adapters; then run the "Load fine-tuned model" cell to use them in the app.

**Requires:** GPU recommended (e.g. Colab T4/A100). On CPU training is very slow.

In [ ]:
# Training config for QLoRA
SENTIMENT_MODEL_NAME = "cardiffnlp/twitter-roberta-base-sentiment-latest"
ADAPTER_SAVE_PATH = "./sentiment_adapters_nigeria"  # where to save LoRA adapters
TRAIN_VAL_SPLIT = 0.15  # fraction for validation
MAX_LENGTH = 128
LORA_R = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.1
BATCH_SIZE = 8
NUM_EPOCHS = 3
LR = 2e-4
USE_4BIT = True  # True = QLoRA (4-bit, GPU). On CPU-only set False (LoRA only).

In [ ]:
# Prepare dataset for training: use labeled data from df_analyzed
from datasets import Dataset, DatasetDict
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer

# Label to id (must match model config; this model uses negative/neutral/positive)
LABEL2ID = {"negative": 0, "neutral": 1, "positive": 2}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}

def prepare_finetune_dataset(df: pd.DataFrame, tokenizer, val_ratio: float = TRAIN_VAL_SPLIT):
    df = df[df["sentiment"].isin(LABEL2ID)].copy()
    df["label"] = df["sentiment"].map(LABEL2ID)
    # Train/val split
   
    train_df, val_df = train_test_split(df, test_size=val_ratio, random_state=42, stratify=df["label"])
    train_ds = Dataset.from_pandas(train_df[["text", "label"]])
    val_ds = Dataset.from_pandas(val_df[["text", "label"]])

    def tokenize(examples):
        return tokenizer(
            examples["text"],
            truncation=True,
            max_length=MAX_LENGTH,
            padding="max_length",
        )

    train_ds = train_ds.map(tokenize, batched=True, remove_columns=["text"])
    val_ds = val_ds.map(tokenize, batched=True, remove_columns=["text"])
    train_ds = train_ds.rename_column("label", "labels")
    val_ds = val_ds.rename_column("label", "labels")
    train_ds.set_format("torch")
    val_ds.set_format("torch")
    return train_ds, val_ds


ft_tokenizer = AutoTokenizer.from_pretrained(SENTIMENT_MODEL_NAME)
train_dataset, val_dataset = prepare_finetune_dataset(df_analyzed, ft_tokenizer, TRAIN_VAL_SPLIT)
print(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}")

In [ ]:
# Load base model with optional 4-bit quantization (QLoRA) and attach LoRA
import torch
from transformers import AutoModelForSequenceClassification, BitsAndBytesConfig, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training

bnb_config = None
if USE_4BIT and torch.cuda.is_available():
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        llm_int8_skip_modules=["classifier"],
    )

ft_model = AutoModelForSequenceClassification.from_pretrained(
    SENTIMENT_MODEL_NAME,
    num_labels=3,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
    quantization_config=bnb_config,
)

# Required for QLoRA: prepare quantized model so trainable adapters can be attached
if bnb_config is not None:
    ft_model = prepare_model_for_kbit_training(ft_model)

lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    target_modules=["query", "value"],  # RoBERTa attention
)
ft_model = get_peft_model(ft_model, lora_config)
ft_model.print_trainable_parameters()

In [ ]:
# Train with HuggingFace Trainer
training_args = TrainingArguments(
    output_dir=ADAPTER_SAVE_PATH,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LR,
    warmup_ratio=0.1,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    fp16=torch.cuda.is_available(),
    report_to="none",
)

trainer = Trainer(
    model=ft_model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
)
trainer.train()
ft_model.save_pretrained(ADAPTER_SAVE_PATH)
ft_tokenizer.save_pretrained(ADAPTER_SAVE_PATH)
print(f"Adapters and tokenizer saved to {ADAPTER_SAVE_PATH}")

### Load fine-tuned adapters (run after training)

Run this cell to switch to the fine-tuned model. Then re-run the **corpus analysis** cell (`df_analyzed = analyze_corpus(...)`) and the **Gradio** cell to refresh results and the UI with the fine-tuned model.

In [ ]:
# Load base model + LoRA adapters and replace the default pipeline
import torch
from peft import PeftModel
from transformers import AutoModelForSequenceClassification, AutoTokenizer

if os.path.isdir(ADAPTER_SAVE_PATH):
    base = AutoModelForSequenceClassification.from_pretrained(
        SENTIMENT_MODEL_NAME,
        num_labels=3,
        id2label=ID2LABEL,
        label2id=LABEL2ID,
    )
    ft_model_inference = PeftModel.from_pretrained(base, ADAPTER_SAVE_PATH)
    ft_model_inference.eval()
    tok = AutoTokenizer.from_pretrained(ADAPTER_SAVE_PATH)
    sentiment_pipeline = pipeline(
        "sentiment-analysis",
        model=ft_model_inference,
        tokenizer=tok,
        device=0 if torch.cuda.is_available() else -1,
        truncation=True,
        max_length=MAX_LENGTH,
    )
    print("Using fine-tuned sentiment model.")
else:
    print("No adapters found at", ADAPTER_SAVE_PATH, "— using base model. Run QLoRA training first.")

## Gradio UI

Select a party to see sentiment distribution and sample tweets. You can also analyze your own tweet.

In [ ]:
def get_party_summary(party: str) -> tuple[str, pd.DataFrame]:
    """Return markdown summary and a DataFrame for BarPlot for the selected party."""
    if party == "All":
        sub = df_analyzed
    else:
        sub = df_analyzed[df_analyzed["party"] == party]
    counts = sub["sentiment"].value_counts()
    total = len(sub)
    lines = [f"**{party}** — {total} tweets", ""]
    for sent in ["positive", "neutral", "negative"]:
        if sent in counts.index:
            pct = 100 * counts[sent] / total
            lines.append(f"- {sent.capitalize()}: {counts[sent]} ({pct:.0f}%)")
    summary_md = "\\n".join(lines)
    # DataFrame for gr.BarPlot: x, y, and optional color
    plot_df = pd.DataFrame({"sentiment": counts.index, "count": counts.values})
    return summary_md, plot_df

def get_sample_tweets(party: str, sentiment_filter: str) -> str:
    """Return sample tweets for party and optional sentiment filter."""
    if party == "All":
        sub = df_analyzed
    else:
        sub = df_analyzed[df_analyzed["party"] == party]
    if sentiment_filter != "All":
        sub = sub[sub["sentiment"] == sentiment_filter]
    if sub.empty:
        return "_No tweets match._"
    sample = sub.sample(min(5, len(sub)))[["text", "sentiment"]]
    try:
        return sample.to_markdown(index=False)
    except AttributeError:
        return sample.to_string(index=False)

In [ ]:
def analyze_custom_tweet(tweet: str) -> str:
    if not (tweet and tweet.strip()):
        return "Enter a tweet to analyze."
    label = get_sentiment(tweet.strip())
    return f"**Sentiment:** {label.capitalize()}"

def update_party_view(party: str):
    summary_md, plot_df = get_party_summary(party)
    return summary_md, plot_df

with gr.Blocks(title="Nigeria Political Sentiment") as app:
    gr.Markdown("## 🇳🇬 Sentiment by party (APC, PDP, LP)")
    party_dd = gr.Dropdown(choices=["APC", "PDP", "LP", "All"], value="APC", label="Party")
    summary_out = gr.Markdown(label="Summary")
    bar_plot = gr.BarPlot(
        value=update_party_view("APC")[1],
        x="sentiment",
        y="count",
        title="Sentiment distribution",
        x_title="Sentiment",
        y_title="Count",
    )
    party_dd.change(
        fn=update_party_view,
        inputs=[party_dd],
        outputs=[summary_out, bar_plot],
    )
    gr.Markdown("---\n### Sample tweets")
    sentiment_filter = gr.Dropdown(choices=["All", "positive", "neutral", "negative"], value="All", label="Sentiment")
    sample_out = gr.Markdown(label="Samples")

    def show_samples(party: str, sent: str):
        return get_sample_tweets(party, sent)

    party_dd.change(fn=show_samples, inputs=[party_dd, sentiment_filter], outputs=[sample_out])
    sentiment_filter.change(fn=show_samples, inputs=[party_dd, sentiment_filter], outputs=[sample_out])

    # Set initial summary, plot, and sample tweets on load
    app.load(
        fn=lambda: (update_party_view("APC")[0], update_party_view("APC")[1], show_samples("APC", "All")),
        outputs=[summary_out, bar_plot, sample_out],
    )

    gr.Markdown("---\n### Analyze your own tweet")
    custom_in = gr.Textbox(placeholder="Paste a tweet about APC, PDP, or LP...", label="Tweet")
    analyze_btn = gr.Button("Analyze sentiment")
    custom_out = gr.Markdown(label="Result")
    analyze_btn.click(fn=analyze_custom_tweet, inputs=[custom_in], outputs=[custom_out])

app.launch()